In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [ ]:
VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"

In [ ]:
import subprocess, sys

VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"


def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)


pip_install(
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"bitsandbytes=={BITSANDBYTES_PIN}",
)

print("profiling pins installed")

installing: transformers==4.46.* accelerate==1.1.* bitsandbytes==0.49.2
profiling pins installed


In [ ]:
import csv
import threading
import time
import subprocess

GPU_SAMPLES = "/content/gpu_samples.csv"

_sampler = {
    "thread": None,
    "stop": None
}


def _sample_loop(stop_event, path, interval_s):

    with open(path, "w", newline="") as fh:

        w = csv.writer(fh)

        w.writerow([
            "t",
            "util_gpu",
            "mem_used_mib"
        ])

        t0 = time.time()

        while not stop_event.is_set():

            out = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-gpu=utilization.gpu,memory.used",
                    "--format=csv,noheader,nounits"
                ],
                capture_output=True,
                text=True,
            ).stdout.strip()


            parts = [
                p.strip()
                for p in out.split(",")
            ]


            if len(parts) == 2:

                w.writerow(
                    [
                        round(time.time()-t0,2),
                        parts[0],
                        parts[1]
                    ]
                )

                fh.flush()


            stop_event.wait(interval_s)



def start_sampler(path=GPU_SAMPLES, interval_s=2):

    if _sampler["thread"] and _sampler["thread"].is_alive():

        print("sampler already running")

        return


    stop = threading.Event()


    th = threading.Thread(
        target=_sample_loop,
        args=(stop,path,interval_s),
        daemon=True
    )


    th.start()


    _sampler["thread"] = th
    _sampler["stop"] = stop


    print("sampler started")



def stop_sampler():

    if _sampler["stop"]:

        _sampler["stop"].set()


    if _sampler["thread"]:

        _sampler["thread"].join(timeout=5)


    _sampler["thread"] = None
    _sampler["stop"] = None


    print("sampler stopped")



def read_util_mean(path=GPU_SAMPLES):

    vals = []


    with open(path) as fh:

        for row in csv.DictReader(fh):

            vals.append(
                float(row["util_gpu"])
            )


    return sum(vals)/len(vals) if vals else 0.0

In [ ]:
import time
import gc
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def load(dtype):

    if dtype == "fp16":

        return AutoModelForCausalLM.from_pretrained(
            MODEL,
            torch_dtype=torch.float16,
            device_map="cuda"
        )


    if dtype == "int8":

        qc = BitsAndBytesConfig(
            load_in_8bit=True
        )

        return AutoModelForCausalLM.from_pretrained(
            MODEL,
            quantization_config=qc,
            device_map="cuda"
        )

In [ ]:
def make_prompt(context_tokens):

    base = "Summarise the following text in one sentence.\n"

    filler = (
        "The data center runs many small inference requests all day. "
        *400
    )

    ids = tok(
        base + filler
    )["input_ids"][:context_tokens]

    return tok.decode(ids)

In [ ]:
def resident_vram_gb():

    torch.cuda.synchronize()

    return torch.cuda.memory_reserved()/(1024**3)

In [ ]:
def profile(model, dtype, context, new_tokens=128, batch=1):

    prompt = make_prompt(context)

    prompts = [prompt] * batch

    enc = tok(
        prompts,
        return_tensors="pt",
        padding=True
    ).to("cuda")


    model.generate(
        **enc,
        max_new_tokens=8,
        do_sample=False
    )


    vram = resident_vram_gb()


    start_sampler()

    t0 = time.time()


    out = model.generate(
        **enc,
        max_new_tokens=new_tokens,
        do_sample=False
    )


    dt = time.time() - t0


    stop_sampler()


    gen_tokens = (
        out.shape[1] -
        enc["input_ids"].shape[1]
    ) * batch


    return {
        "dtype": dtype,
        "context": context,
        "vram_gb": round(vram,3),
        "util_mean": round(read_util_mean(),1),
        "tokens_per_s": round(gen_tokens/dt,1)
    }

In [ ]:
rows=[]


for dtype in ["fp16","int8"]:

    model=load(dtype)


    for context in [512,2048,4096]:

        row=profile(
            model,
            dtype,
            context,
            new_tokens=128,
            batch=1
        )

        print(row)

        rows.append(row)


    del model

    gc.collect()

    torch.cuda.empty_cache()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started
sampler stopped
{'dtype': 'fp16', 'context': 512, 'vram_gb': 3.113, 'util_mean': 40.0, 'tokens_per_s': 25.6}


...

sampler started
sampler stopped
{'dtype': 'int8', 'context': 512, 'vram_gb': 1.805, 'util_mean': 25.9, 'tokens_per_s': 5.9}


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started
sampler stopped
{'dtype': 'int8', 'context': 2048, 'vram_gb': 2.035, 'util_mean': 27.2, 'tokens_per_s': 5.7}


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started
sampler stopped
{'dtype': 'int8', 'context': 4096, 'vram_gb': 2.309, 'util_mean': 32.2, 'tokens_per_s': 5.4}


In [ ]:
import json

with open("profile.json","w") as f:
    json.dump(rows,f,indent=2)

print("saved")

saved


In [ ]:
model = load("fp16")


b1 = profile(
    model,
    "fp16",
    512,
    new_tokens=128,
    batch=1
)


b8 = profile(
    model,
    "fp16",
    512,
    new_tokens=128,
    batch=8
)


del model

gc.collect()

torch.cuda.empty_cache()


print("batch 1:", b1)
print("batch 8:", b8)


print(
    "tokens/s ratio:",
    round(
        b8["tokens_per_s"] / b1["tokens_per_s"],
        2
    )
)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started
sampler stopped


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started
sampler stopped
batch 1: {'dtype': 'fp16', 'context': 512, 'vram_gb': 3.113, 'util_mean': 57.0, 'tokens_per_s': 26.8}
batch 8: {'dtype': 'fp16', 'context': 512, 'vram_gb': 3.529, 'util_mean': 76.3, 'tokens_per_s': 216.9}
tokens/s ratio: 8.09


In [ ]:
import json


with open("batch_check.json","w") as f:

    json.dump(
        {
            "batch1_tokens_per_s": b1["tokens_per_s"],
            "batch8_tokens_per_s": b8["tokens_per_s"]
        },
        f,
        indent=2
    )


print("batch_check.json saved")

batch_check.json saved


In [ ]:
%run "verify_cell (1).py"

rows: 6, dtypes: ['fp16', 'int8'], contexts: [512, 2048, 4096]
batch-1 tokens/s: 26.8, batch-8 tokens/s: 216.9
GREEN CHECK: PASS
